## 1. Assignment Tasks

1) Define a problem statement
__A) Can flight and different factors be used to predict total arrival delay hours for an airline/airport/month?__
2) Perform EDA (if you are choosing a new dataset) 
3) Use EDA Insights for feature selection and feature engineering
4) Create your first 3 models using the framework
5) Evaluate the model and then attempt to improve it
6) Interpret Model results and outputs (coefficients, trees) and continue to add to your insights.

## 2. Predictive Modelling

### Cleaning & Pre-Processing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo 
from sklearn import datasets

# Preprocessing
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Evaluation metrics
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, root_mean_squared_error, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

# For combining pipelines after encoding
from sklearn.compose import make_column_selector as selector

sns.set(style="whitegrid")

In [ ]:
df = pd.read_csv("Airline_Delay_Cause.csv")

Some Basic Cleaning

In [ ]:
# Clean column names
df.columns = df.columns.str.strip().str.lower()
df.columns = df.columns.str.replace(" ", "_")

Generating Continuous Data to use regression

In [ ]:
# 1. Define features and target
features = [
    'arr_flights',
    'arr_del15',
    'carrier_ct',
    'weather_ct',
    'nas_ct',
    'security_ct',
    'late_aircraft_ct',
    'arr_cancelled',
    'arr_diverted',
    'carrier',
    'airport',
    'month'
]

target = 'arr_delay'

# 2. Define feature types
num_features = [
    'arr_flights',
    'arr_del15',
    'carrier_ct',
    'weather_ct',
    'nas_ct',
    'security_ct',
    'late_aircraft_ct',
    'arr_cancelled',
    'arr_diverted'
]

cat_features = ['carrier', 'airport', 'month']



In [ ]:
# 3. Handle missing values
df = df.dropna(subset=['arr_delay'])

df[num_features] = df[num_features].fillna(df[num_features].median())
df[cat_features] = df[cat_features].fillna("Unknown")

# 4. Create X and y
X = df[features]
y = df[target]

In [ ]:
df['arr_delay'] = np.log1p(df['arr_delay'])
#df.fillna(0, inplace = True) #log transform might create nan values

In [ ]:
# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ],
    sparse_threshold=0
)

# Apply preprocessing to feature set
X_processed = preprocessor.fit_transform(X)

# Get transformed feature names
new_feature_names = (
    num_features +
    list(preprocessor.named_transformers_['cat'].get_feature_names_out(cat_features))
)

# Turn processed data into a DataFrame
df_transformed = pd.DataFrame(X_processed, columns=new_feature_names)

print(df_transformed.head())

In [ ]:
# Use processed features
X_processed = df_transformed

# Target
y_reg = y

# Train-test split (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_reg, test_size=0.3, random_state=42
)

### A. Regression Models

In [ ]:
# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred_lin = lin_reg.predict(X_test)
r2_lin = r2_score(y_test, y_pred_lin)
mse_lin = mean_squared_error(y_test, y_pred_lin)
mae_lin = mean_absolute_error(y_test, y_pred_lin)

print("Linear Regression R²:", r2_lin, "MSE:", mse_lin, "MAE:", mae_lin)

In [ ]:

# Decision Tree Regressor
dt_reg = DecisionTreeRegressor(random_state=42)
dt_reg.fit(X_train, y_train)
y_pred_dt = dt_reg.predict(X_test)
r2_dt = r2_score(y_test, y_pred_dt)
mse_dt = mean_squared_error(y_test, y_pred_dt)
print("Decision Tree Regressor R²:", r2_dt, "MSE:", mse_dt)

In [ ]:
# Random Forest Regressor

rf_reg = RandomForestRegressor(n_estimators=10, random_state=42)
rf_reg.fit(X_train, y_train)
y_pred_rf = rf_reg.predict(X_test)
r2_rf = r2_score(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
print("Random Forest Regressor R²:", r2_rf, "MSE:", mse_rf)

Ridge Regression and Gradient Boosting were used to improve the model’s performance. Ridge Regression helps when there are a lot of features by slightly reducing the impact of each one, which prevents the model from overfitting. This is useful because the dataset has many variables after encoding things like airports and carriers.

Gradient Boosting was also used because it can capture more complex patterns in the data. It builds multiple decision trees step-by-step, where each new tree focuses on fixing the mistakes of the previous ones. This allows the model to better understand how different factors contribute to flight delays

In [ ]:
# Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

y_pred_ridge = ridge.predict(X_test)

print("Ridge R²:", r2_score(y_test, y_pred_ridge))
print("Ridge MSE:", mean_squared_error(y_test, y_pred_ridge))

In [ ]:
# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)

print("Gradient Boosting R²:", r2_score(y_test, y_pred_gb))
print("Gradient Boosting MSE:", mean_squared_error(y_test, y_pred_gb))

### B. Interpreting Results

#### Regression Coefficients

In [ ]:
# Regression coefficients from Linear Regression
coefficients = lin_reg.coef_
coef_series = pd.Series(coefficients, index=X_train.columns)

top_positive = coef_series.sort_values(ascending=False).head(10)
top_negative = coef_series.sort_values().head(10)
delay_only = coef_series[coef_series.index.str.contains("ct|delay", case=False)].sort_values(ascending=False)

print("Top 10 Positive Regression Coefficients:")
print(top_positive)

print("\nTop 10 Negative Regression Coefficients:")
print(top_negative)

print("\nDelay-Related Coefficients:")
print(delay_only)

The coefficients are interpreted as "the change in the target associated with a unit change in that variable, if all else remains the same". 
- A higher positive value means the feature has a strong positive association with the target.
- A negative value means the feature is inversely associated with the target.
- This helps in understanding which predictors have the most influence on your continuous target variable.

#### Visualizing a Decision Tree


In [ ]:
dt_reg_small = DecisionTreeRegressor(max_depth=3)
dt_reg_small.fit(X_train, y_train)

plt.figure(figsize=(10, 6))
plot_tree(
    dt_reg_small,
    feature_names=X_train.columns,
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3,
    impurity=False
)
plt.show()

#### Hyperparameter Tuning 

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

# make sure y is 1-dimensional
if hasattr(y_train, "values"):
    y_train_fixed = y_train.values.ravel()
else:
    y_train_fixed = y_train.ravel()

dt_reg = DecisionTreeRegressor(random_state=42)

param_grid = {
    'max_depth': [3, 5],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    estimator=dt_reg,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=1
)

grid_search.fit(X_train, y_train_fixed)

print("Best Hyperparameters:", grid_search.best_params_)
print("Best CV R2:", grid_search.best_score_)

In [ ]:
results_df = pd.DataFrame(grid_search.cv_results_)

results_df = results_df[['params', 'mean_test_score', 'std_test_score']]

results_df = results_df.sort_values(by='mean_test_score', ascending=False)

results_df.head(10)